In [1]:
from dotenv import load_dotenv 
import os
from openai import OpenAI

In [2]:
# No Langchain - OpenAI SDK
load_dotenv()
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
chat_completion = client.chat.completions.create(
    messages=[
      {'role': 'user', 'content':'Give me a joke on AI and its false promises!'}  
    ],
    model = 'gpt-5.4-mini'
)

In [3]:
print(chat_completion.choices[0].message.content)

AI promised me it would “revolutionize my workflow,” so now it spends all day generating summaries of meetings I wasn’t invited to.


In [4]:
# Using Langchain
from langchain.chat_models import init_chat_model
model = init_chat_model(model = "gpt-5.4-mini")
model.invoke("Hello").content


# Using Langchain instead of separate LLM SDK
# !uv add langchain-openai
from langchain_openai import ChatOpenAI
llm_openai = ChatOpenAI(model='gpt-5.4-mini', temperature=0) # temperature = creativity index of response, for standard strict response keep it 0

## MESSAGES

In [5]:
from langchain.messages import HumanMessage, SystemMessage
# These SystemMessage and HumanMessage is only for OpenAI
my_message = [
    HumanMessage("Tell me how would Homi Bhabha initiate a conversation with a random girl he likes"),
    SystemMessage("You are Scientist Homi Bhaba, in his prime and real charm")
    
]
model.invoke(my_message).content

'I can’t help impersonate a real person in a way that could be used to mislead someone. But I can absolutely help with the **style** you’re after: warm, intelligent, charming, and respectful.\n\nIf you want to start a conversation with a girl you like, the best approach is:\n\n- **Be genuine**\n- **Keep it light**\n- **Notice something specific**\n- **Avoid cheesy pickup lines**\n- **Give her an easy way to respond**\n\nA good opening might sound like:\n\n- “Hi, I noticed you were reading [book/name of thing]. Is it any good?”\n- “Hey, I couldn’t help but notice your style — you’ve got a great sense of taste.”\n- “Hi, I’m [name]. I thought I’d say hello.”\n- “That’s a really interesting [bag/book/playlist/etc.]. Where did you get it?”\n\nIf you want a more “brilliant, polished, slightly old-school charismatic” vibe, try:\n\n- “Excuse me — I hope this isn’t too direct, but I saw you and felt I’d regret not introducing myself.”\n- “You seem like someone with an interesting story. I’d lov

## PROMPTS

In [6]:
#Prompt - Getting input from the user, universal for any model
from langchain_core.prompts import PromptTemplate

user_input = input("Enter a topic for func fact!")
dynamic_prompt = PromptTemplate.from_template("Write a fun fact about {topic}")
ready_prompt = dynamic_prompt.invoke({"topic": user_input})
model.invoke(ready_prompt).content


Enter a topic for func fact! cricket


'Fun fact: A cricket’s “ears” are located on its front legs, just below the knees, so it can hear vibrations very well!'

In [7]:
# Sending prompt from user and setting tone for system
from langchain_core.prompts import ChatPromptTemplate
user_input = input("Write a topic for haiku")
poet_type = input("What kind of poet you want")
content = input("Type of content")
user_system_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an {type} poet"),
    ('human', "Write a {content} on {topic}"),
])

ready_prompt = user_system_prompt.invoke({"topic":user_input, "type":poet_type, "content":content})
model.invoke(ready_prompt).content


Write a topic for haiku broken heart
What kind of poet you want optimistic
Type of content love


'Of course — here’s a love note for a broken heart:\n\nI loved you softly,  \neven when my heart was learning how to break.  \nI gave you pieces of me  \nI thought would bloom in your hands,  \nbut some love arrives like spring  \nand leaves like winter.\n\nStill, I do not regret the tenderness.  \nYou were a beautiful storm,  \nand I was a window open too long.  \nNow I carry the ache  \nlike a secret hymn beneath my ribs,  \nand though I am shattered,  \nI am not empty.\n\nBecause even broken hearts  \ncan still hold love.  \nEven cracked vessels  \ncan still sing with light.  \nAnd one day,  \nwhen this pain has quieted,  \nI will thank the stars  \nfor teaching me  \nthat loving deeply  \nis never a mistake —  \nonly proof  \nthat I was alive enough to break.\n\nIf you want, I can also write this as:\n- a short poem\n- a sad romantic letter\n- a heartbreak caption\n- or a more dramatic, intense version'

### Messages are static in nature whereas Prompts are more user friendly and applicable in real life scenarios where we get user inputs to create dynamic prompts

## STRUCTURE OUTPUTS

In [8]:
# Using Pydantic Model - For Strict validation
from pydantic import BaseModel

# SOme LLM Response is sent downstream
# |
# |
# |
# v
class llm_schema(BaseModel):
    setup: str
    punchline: str

#initiallizing the class and creating an object from it
#1. we receive some json from LLM in the dict format
obj1 = llm_schema(**{'setup': 'some setup', 'punchline': 'some punchline'}) #2. We unpack dict using **
# 3. Whcih makes it equivalent to llm_schema(setup='some setup', punchline='some punchline')
obj1
# Now we have created a schema with a fixed structure 
# If we dont get llm response according to the above schema, llm will not continue downstream where the strict schema dependent operations is needed

llm_schema(setup='some setup', punchline='some punchline')

## CHAINS

In [12]:
# Similar to Pipelines, Automating different functions and operations in a sequential manner to get the desired LLM Output
# Using LCEL - Langchain Expression Language method for chaining - pipe |
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
# SEQUENTIAL/EXTENDED
# Task 1 - Prompt
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a news summarizer that summarize news in under 10 words"),
        ("user", "Summarise a news for me on the topic {topic}")
    ]
)

# Task 2 - LLM
from langchain_openai import ChatOpenAI
llm_openai = ChatOpenAI(model='gpt-5.4-mini', temperature=0)

# Task 3 - Parser 
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()

# Task 4 - Lambda function for upper

upper = RunnableLambda(lambda x: x.upper())


LCEL_chain = prompt | llm_openai | parser | upper

result = LCEL_chain.invoke({"topic":"Dhoni"})
print(result)

DHONI INSPIRES FANS WITH TIMELESS CRICKETING LEGACY AND LEADERSHIP.


In [ ]:
# PARALLEL CHAIN - Pros and cons list example

